# Choose frames to annotate — many matches, few frames each

Scans every match video in a Drive folder and picks a small, varied, hard set of frames
for annotation in Roboflow, with the current detector's boxes as pre-labels (correct them,
don't draw from scratch).

**How frames are chosen**
1. One candidate per second per match; only frames with almost no pitch (crowd close-ups,
   graphics, studio) and the blurriest frames are dropped. Zoomed-out shots during play stay:
   they hold the smallest players, where the detector is weakest.
2. Farthest-point sampling in an appearance space, with a per-match quota and a minimum
   gap in time — a new stadium, kit or light counts more than another minute of a match
   already covered.
3. The current detector scores a shortlist: frames with uncertain boxes, small distant
   players and crowded groups rank higher (that is where it fails).
4. Whole matches are held out for validation, so the score is never inflated by
   near-identical frames on both sides.

**Before running**
- `Runtime → Change runtime type → T4 GPU` (CPU works, the detector step is just slower).
- Colab Secrets: `ROBOFLOW_API_KEY`.
- Put match videos in `MyDrive/Playbook/annotation_pool/` — one file per match, named after
  the match (e.g. `HILAL-AHLI_2025-10-03.mp4`). Different matches matter more than long ones.

**Output** (in `MyDrive/Playbook/annotation_pool/_selection_<timestamp>/`): `train.zip` and
`valid.zip` in COCO format with pre-labels, a contact sheet of every chosen frame, and a CSV.
Upload each zip to Roboflow `players-detection-my09y` → Upload, choosing the split to match.

**Already ran it and only want to change the split?** Set `VALID_MATCHES` in cell 4, re-run
cell 4, then cells 9 and 10 (the detector step does not need to run again while the runtime
is still connected).

In [ ]:
# Cell 1 — GPU check (optional: CPU works, slower)
import subprocess
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
USE_GPU = g.returncode == 0
print('GPU:', g.stdout.strip() if USE_GPU else 'none — the detector step will run on CPU')

In [ ]:
# Cell 2 — Install
!pip uninstall -qqy opencv-python opencv-python-headless >/dev/null 2>&1
!pip install -q 'numpy>=2.0.0,<2.4.0' opencv-python-headless==4.10.0.84 'supervision==0.27.0.post2' 'pandas>=2.0'
if USE_GPU:
    !pip install -q inference-gpu==1.3.0
    !pip install -q onnxruntime-gpu==1.20.1 --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
else:
    !pip install -q inference==1.2.2
print('Installed.')

In [ ]:
# Cell 3 — Clone or update the repo
import os, sys
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
DEST = '/content/playbook'
if os.path.isdir(DEST + '/.git'):
    !git -C {DEST} fetch -q origin {BRANCH}
    !git -C {DEST} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} https://github.com/muwafagq/playbook-program.git {DEST}
os.chdir(DEST); sys.path.insert(0, DEST)
from tools import frame_sampler as fs
print('Ready.')

In [ ]:
# Cell 4 — Settings
POOL_DIR = '/content/drive/MyDrive/Playbook/annotation_pool'
TARGET_FRAMES = 300        # total to annotate across all matches (300-500 solid, 800-1000 strong)
MIN_PER_MATCH, MAX_PER_MATCH = 5, 40
EVERY_S = 1.0              # candidate spacing
MIN_GAP_S = 4.0            # minimum time between two picks from one match
SKIP_START_S = 0.0         # e.g. skip pre-match studio footage
POOL_FACTOR = 4            # shortlist size per match for the detector step
DETECTOR = 'footballs-player-detection-zkams-zia6c/2'   # current pipeline detector
VALID_FRAC = 0.2           # share of matches held out for validation (used if VALID_MATCHES is empty)
VALID_MATCHES = []         # match names to hold out, e.g. two NIGHT games (the target condition);
                           # names as printed by cell 5. Empty = random choice.

In [ ]:
# Cell 5 — Mount Drive, list matches
import glob, re, math
from google.colab import drive
drive.mount('/content/drive')
videos = sorted(p for ext in ('mp4', 'mkv', 'mov', 'avi', 'MP4', 'MOV')
                for p in glob.glob(f'{POOL_DIR}/**/*.{ext}', recursive=True) if '/_selection_' not in p)
if not videos:
    raise SystemExit(f'No videos found in {POOL_DIR}')
match_of = {v: re.sub(r'[^A-Za-z0-9_-]+', '_', os.path.splitext(os.path.basename(v))[0]) for v in videos}
PER_MATCH = max(MIN_PER_MATCH, min(MAX_PER_MATCH, math.ceil(TARGET_FRAMES / len(videos))))
print(f'{len(videos)} matches -> {PER_MATCH} frames each (~{PER_MATCH * len(videos)} total)')
for v in videos:
    print('  ', match_of[v])

In [ ]:
# Cell 6 — Scan: one candidate per second, keep game-view shots
cands = []
for v in videos:
    c = fs.scan_video(v, match_of[v], every_s=EVERY_S, skip_start_s=SKIP_START_S)
    k = fs.filter_broadcast(c)
    cands += k
    print(f'{match_of[v]:40s} scanned {len(c):5d}, game-view kept {len(k):5d}', flush=True)

In [ ]:
# Cell 7 — Shortlist for the detector (diverse, POOL_FACTOR x the final quota)
pool = fs.select_diverse(cands, per_match=PER_MATCH * POOL_FACTOR, min_gap_s=MIN_GAP_S)
print(f'Shortlist: {len(pool)} frames')

In [ ]:
# Cell 8 — Current detector on the shortlist: pre-labels + hardness
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
from inference import get_model
import supervision as sv
model = get_model(model_id=DETECTOR, api_key=os.environ['ROBOFLOW_API_KEY'])
for i, c in enumerate(pool):
    img = fs.read_frame(c.video, c.frame)
    det = sv.Detections.from_inference(model.infer(img, confidence=0.10)[0])
    c.detections = [(*b, float(s), int(k)) for b, s, k in zip(det.xyxy.tolist(), det.confidence.tolist(), det.class_id.tolist())]
    c.hardness = fs.hardness(c.detections, img.shape[0])
    if i % 50 == 0:
        print(f'  {i}/{len(pool)}', flush=True)
print('Detector done.')

In [ ]:
# Cell 9 — Final pick: diverse and hard, whole matches held out for validation
import pandas as pd
picks = fs.select_diverse(pool, per_match=PER_MATCH, min_gap_s=MIN_GAP_S, hardness_weight=1.0)
splits = fs.split_matches(list(match_of.values()), valid_frac=VALID_FRAC, valid_matches=VALID_MATCHES)
df = pd.DataFrame([{'match': c.match, 'split': splits[c.match], 'frame': c.frame, 'time_s': round(c.time_s, 1),
                    'hardness': round(c.hardness, 2), 'prelabels': sum(d[4] >= 0.3 for d in c.detections)} for c in picks])
print(df.groupby(['split', 'match']).agg(frames=('frame', 'size'), mean_hardness=('hardness', 'mean')).round(2).to_string())

In [ ]:
# Cell 10 — Export to Drive: train.zip / valid.zip (COCO + pre-labels), contact sheet, CSV
import time
from IPython.display import Image, display
out = f"{POOL_DIR}/_selection_{time.strftime('%Y%m%d_%H%M')}"
zips = fs.export_coco(picks, out, splits)
df.to_csv(f'{out}/selection.csv', index=False)
fs.contact_sheet(picks, f'{out}/contact_sheet.jpg')
for split, zp in zips.items():
    print(f'{split}: {zp} ({os.path.getsize(zp) / 1e6:.1f} MB)')
print('Upload each zip to Roboflow players-detection-my09y -> Upload, choosing the matching split.')
display(Image(f'{out}/contact_sheet.jpg', width=1100))